# Real-Hardware QAOA Training Notebook

This notebook trains QAOA angles on IBM Quantum hardware. It is separate from the benchmark/training notebooks on purpose.

Important distinction:

- The winning simulation method is the valid-subspace QAOA candidate generator.
- This hardware notebook trains the hardware-executable full-binary QUBO bridge.
- The objective used for training is measured scaled Ising/QUBO energy from IBM counts.
- Projected AR is reported as a diagnostic, not used as the training loss.

Default behavior is safe: it can connect and transpile, but it will not submit training jobs until `RUN_HARDWARE_TRAINING = True`.


## 1. Imports And Configuration

The default target is the smaller 18-qubit scenario that already ran on `ibm_quebec`:

- `U=3, G=6, S=5`, seed `35`
- previous ISA depth: `882`
- previous two-qubit gate count: `780` CZ gates

One SPSA training step submits one IBM job containing two circuits: one `theta_plus`, one `theta_minus`.


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import importlib
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

from IPython.display import Markdown, display

import qaoa_isac_benchmark as benchmark
import qaoa_isac_env as env_module

importlib.invalidate_caches()
benchmark = importlib.reload(benchmark)
env_module = importlib.reload(env_module)

SystemParams = env_module.SystemParams
build_environment = benchmark.build_environment
build_full_binary_qaoa_circuit = benchmark.build_full_binary_qaoa_circuit
circuit_summary = benchmark.circuit_summary
enumerate_assignments = benchmark.enumerate_assignments
extract_sampler_counts = benchmark.extract_sampler_counts
scaled_ising_terms = benchmark.scaled_ising_terms
summarize_random_bitstring_projection_baseline = benchmark.summarize_random_bitstring_projection_baseline

# IBM setup. Leave IBM_INSTANCE as None unless your account requires a CRN/instance string.
IBM_INSTANCE = None
FIXED_BACKEND_NAME = None  # Example: "ibm_quebec". None means least_busy operational backend.

# Safe execution gates.
CONNECT_TO_IBM = True
CHECK_TRANSPILATION = True
RUN_HARDWARE_TRAINING = False
RUN_FINAL_HARDWARE_EVAL = False

# Hardware budget. Increase only after a smoke run works.
TRAINING_SHOTS = 256
FINAL_SHOTS = 1024
SPSA_STEPS = 6
SPSA_SEED = 260630

# SPSA hyperparameters for two angles: gamma and beta.
SPSA_A0 = 0.22
SPSA_C0 = 0.18
SPSA_ALPHA = 0.602
SPSA_GAMMA = 0.101

# Submit only if the transpilation is not much worse than the previous measured 18-qubit run.
REFERENCE_DEPTH = 882
REFERENCE_TWO_QUBIT_GATES = 780
MAX_DEPTH_FACTOR = 1.20
MAX_TWO_QUBIT_FACTOR = 1.20

OUTPUT_JSON = Path("qaoa_isac_hardware_training_results.json")

# Start from the best known full-binary QUBO-energy simulator angles.
try:
    benchmark_results = json.loads(Path("qaoa_isac_benchmark_results.json").read_text(encoding="utf-8"))
    initial_angles = benchmark_results["hardware_demo_candidate"]["full_binary_angle_probe"]["qubo_energy_optimized_angles"]
    INITIAL_GAMMA = float(initial_angles["gamma"])
    INITIAL_BETA = float(initial_angles["beta"])
except Exception:
    INITIAL_GAMMA = 0.8975979010256552
    INITIAL_BETA = 2.4683942278205517

print(f"Initial hardware-training angles: gamma={INITIAL_GAMMA:.6f}, beta={INITIAL_BETA:.6f}")
print(f"Training will submit {SPSA_STEPS} IBM jobs if RUN_HARDWARE_TRAINING=True.")


## 2. Build The 18-Qubit Hardware Scenario

This builds the same full-binary QUBO hardware bridge used by the completed 18-qubit IBM run.


In [ ]:
HARDWARE_PARAMS = SystemParams(U=3, G=6, S=5, Nt=4, Gamma_min=0.5)
HARDWARE_SEED = 35

hardware_env = build_environment(HARDWARE_PARAMS, seed=HARDWARE_SEED, quiet=True)
feasible_states = enumerate_assignments(hardware_env, require_c3=True)
exact_index, exact_state = max(enumerate(feasible_states), key=lambda item: item[1].sum_rate)
hardware_exact_rate = exact_state.sum_rate

h_scaled, j_scaled, qubo_scale = scaled_ising_terms(hardware_env)

print(f"Full-binary qubits: {hardware_env.n_qubits}")
print(f"Feasible assignments: {len(feasible_states)}")
print(f"Exact assignment: {list(exact_state.assignment)}")
print(f"Exact sum-rate: {hardware_exact_rate / 1e6:.6f} Mbps")
print(f"QUBO scale used in circuit: {qubo_scale:.6e}")

preview_circuit = build_full_binary_qaoa_circuit(
    hardware_env,
    gamma=INITIAL_GAMMA,
    beta=INITIAL_BETA,
    reps=1,
    measure=True,
)
preview_summary = circuit_summary(preview_circuit)
print("Pre-ISA circuit summary:")
print(json.dumps(preview_summary, indent=2))


## 3. Count Processing And Hardware Objective

Training loss is the measured mean scaled Ising energy from raw IBM bitstrings.
Lower is better.

Projected AR is reported only as a diagnostic. It uses nearest feasible assignment projection and the exact feasible optimum for interpretation, but the optimizer does not train directly on the exact label.


In [ ]:
def wrap_angles(theta):
    gamma = float(theta[0] % (2.0 * math.pi))
    beta = float(theta[1] % math.pi)
    return np.array([gamma, beta], dtype=float)


def bitstring_to_variable_vector(bitstring, n_qubits):
    # Qiskit count strings display highest classical bit first.
    return np.array([int(bit) for bit in bitstring[::-1][:n_qubits]], dtype=int)


def assignment_vector(row, env):
    x = np.zeros(env.n_qubits, dtype=int)
    for u, g in enumerate(row.assignment):
        x[u * env.params.G + g] = 1
    return x


feasible_vectors = {tuple(assignment_vector(row, hardware_env)): row for row in feasible_states}
feasible_lookup = {row.assignment: row for row in feasible_states}


def scaled_ising_energy_from_bits(bits):
    z = 1.0 - 2.0 * np.asarray(bits, dtype=float)
    pair_energy = float(np.sum(np.triu(j_scaled, 1) * np.outer(z, z)))
    field_energy = float(z @ h_scaled)
    return field_energy + pair_energy


def project_bitstring_to_feasible_assignment(bitstring):
    bits = bitstring_to_variable_vector(bitstring, hardware_env.n_qubits)
    best_row = None
    best_key = None
    for row in feasible_states:
        candidate = assignment_vector(row, hardware_env)
        hamming = int(np.sum(bits != candidate))
        # First minimize Hamming distance, then prefer better exact sum-rate.
        key = (-hamming, row.sum_rate)
        if best_key is None or key > best_key:
            best_key = key
            best_row = row
    return best_row


def summarize_counts_for_training(counts, theta, job_id, register_name, label, shots):
    total = int(sum(counts.values()))
    if total <= 0:
        raise ValueError("No shots returned from hardware job.")

    raw_feasible_count = 0
    raw_feasible_best = None
    energy_sum = 0.0
    projected_counts = Counter()
    projected_rate_sum = 0.0

    for bitstring, count in counts.items():
        bits = bitstring_to_variable_vector(bitstring, hardware_env.n_qubits)
        energy_sum += int(count) * scaled_ising_energy_from_bits(bits)

        vector = tuple(bits)
        if vector in feasible_vectors:
            raw_feasible_count += int(count)
            row = feasible_vectors[vector]
            if raw_feasible_best is None or row.sum_rate > raw_feasible_best.sum_rate:
                raw_feasible_best = row

        projected = project_bitstring_to_feasible_assignment(bitstring)
        projected_counts[projected.assignment] += int(count)
        projected_rate_sum += int(count) * projected.sum_rate

    best_projected_assignment, best_projected_count = projected_counts.most_common(1)[0]
    best_projected = feasible_lookup[best_projected_assignment]
    projected_optimum_count = sum(
        count
        for assignment, count in projected_counts.items()
        if feasible_lookup[assignment].sum_rate >= hardware_exact_rate - 1e-9
    )

    return {
        "label": label,
        "job_id": job_id,
        "register": register_name,
        "shots": total,
        "requested_shots": int(shots),
        "gamma": float(theta[0]),
        "beta": float(theta[1]),
        "mean_scaled_ising_energy": float(energy_sum / total),
        "raw_feasible_count": int(raw_feasible_count),
        "raw_feasible_rate": float(raw_feasible_count / total),
        "raw_feasible_best_AR_rate": None if raw_feasible_best is None else float(raw_feasible_best.sum_rate / hardware_exact_rate),
        "projected_mean_AR_rate": float(projected_rate_sum / total / hardware_exact_rate),
        "best_projected_assignment": list(best_projected.assignment),
        "best_projected_AR_rate": float(best_projected.sum_rate / hardware_exact_rate),
        "best_projected_count": int(best_projected_count),
        "projected_optimum_count": int(projected_optimum_count),
        "projected_optimum_rate": float(projected_optimum_count / total),
    }


def display_metrics(rows):
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        print(json.dumps(rows, indent=2))


## 4. Connect To IBM And Check Transpilation

This cell does not submit a hardware job. It only selects a backend, builds the pass manager, and checks the compiled circuit depth/gate count.


In [ ]:
service = None
backend = None
pass_manager = None
sampler = None
transpilation_summary = None

if CONNECT_TO_IBM:
    from qiskit.transpiler import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qiskit_ibm_runtime import SamplerV2 as Sampler

    service = QiskitRuntimeService(instance=IBM_INSTANCE) if IBM_INSTANCE else QiskitRuntimeService()
    if FIXED_BACKEND_NAME:
        backend = service.backend(FIXED_BACKEND_NAME)
    else:
        backend = service.least_busy(
            simulator=False,
            operational=True,
            min_num_qubits=hardware_env.n_qubits,
        )
    print(f"Selected backend: {backend.name}")

    pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=1)
    sampler = Sampler(mode=backend)
else:
    print("CONNECT_TO_IBM is False. Set it to True before hardware training.")

if CHECK_TRANSPILATION and pass_manager is not None:
    isa_preview = pass_manager.run(preview_circuit)
    transpilation_summary = {
        "backend": backend.name,
        "depth": int(isa_preview.depth()),
        "ops": {str(name): int(count) for name, count in isa_preview.count_ops().items()},
    }
    transpilation_summary["two_qubit_gate_count"] = int(
        sum(
            count
            for name, count in transpilation_summary["ops"].items()
            if name in {"cx", "cz", "ecr", "swap", "rzz"}
        )
    )
    print(json.dumps(transpilation_summary, indent=2))

    depth_ok = transpilation_summary["depth"] <= REFERENCE_DEPTH * MAX_DEPTH_FACTOR
    twoq_ok = transpilation_summary["two_qubit_gate_count"] <= REFERENCE_TWO_QUBIT_GATES * MAX_TWO_QUBIT_FACTOR
    print(f"Depth acceptable vs reference: {depth_ok}")
    print(f"Two-qubit count acceptable vs reference: {twoq_ok}")
    if not (depth_ok and twoq_ok):
        print("Do not submit yet. Wait for a better backend or lower the circuit cost.")


## 5. Hardware Evaluation Function

This function submits one IBM job for one or more angle pairs and waits for results. For SPSA, each training step sends two circuits in one job.


In [ ]:
hardware_training_evaluations = []

def run_hardware_evaluations(theta_list, shots, label):
    if sampler is None or pass_manager is None:
        raise RuntimeError("IBM sampler/pass_manager is not initialized. Run the backend setup cell first.")

    pubs = []
    clean_thetas = []
    for theta in theta_list:
        theta = wrap_angles(theta)
        circuit = build_full_binary_qaoa_circuit(
            hardware_env,
            gamma=float(theta[0]),
            beta=float(theta[1]),
            reps=1,
            measure=True,
        )
        isa_circuit = pass_manager.run(circuit)
        pubs.append((isa_circuit,))
        clean_thetas.append(theta)

    job = sampler.run(pubs, shots=int(shots))
    print(f"Submitted {label}: job_id={job.job_id()}, circuits={len(pubs)}, shots={shots}")
    result = job.result()
    print(f"Completed {label}: status={job.status()}")

    rows = []
    for index, theta in enumerate(clean_thetas):
        pub_result = result[index]
        counts, register_name = extract_sampler_counts(pub_result, return_register=True)
        row = summarize_counts_for_training(
            counts,
            theta=theta,
            job_id=job.job_id(),
            register_name=register_name,
            label=f"{label}/circuit_{index}",
            shots=shots,
        )
        rows.append(row)
        hardware_training_evaluations.append(row)

    return rows


## 6. Run Real-Hardware SPSA Training

Set `RUN_HARDWARE_TRAINING = True` in Cell 1 before running this cell.

Cost estimate:

```text
number of IBM jobs = SPSA_STEPS
circuits per job = 2
shots per circuit = TRAINING_SHOTS
```


In [ ]:
trained_theta = wrap_angles([INITIAL_GAMMA, INITIAL_BETA])
spsa_log = []

if not RUN_HARDWARE_TRAINING:
    print("RUN_HARDWARE_TRAINING is False. Set it to True to submit IBM training jobs.")
else:
    rng = np.random.default_rng(SPSA_SEED)
    print(f"Starting hardware SPSA from gamma={trained_theta[0]:.6f}, beta={trained_theta[1]:.6f}")

    for step in range(1, SPSA_STEPS + 1):
        ak = SPSA_A0 / (step ** SPSA_ALPHA)
        ck = SPSA_C0 / (step ** SPSA_GAMMA)
        delta = rng.choice([-1.0, 1.0], size=2)

        theta_plus = wrap_angles(trained_theta + ck * delta)
        theta_minus = wrap_angles(trained_theta - ck * delta)

        rows = run_hardware_evaluations(
            [theta_plus, theta_minus],
            shots=TRAINING_SHOTS,
            label=f"spsa_step_{step}",
        )
        loss_plus = rows[0]["mean_scaled_ising_energy"]
        loss_minus = rows[1]["mean_scaled_ising_energy"]
        grad_hat = ((loss_plus - loss_minus) / (2.0 * ck)) * delta
        trained_theta = wrap_angles(trained_theta - ak * grad_hat)

        step_row = {
            "step": step,
            "ak": float(ak),
            "ck": float(ck),
            "delta": [float(value) for value in delta],
            "loss_plus": float(loss_plus),
            "loss_minus": float(loss_minus),
            "grad_hat": [float(value) for value in grad_hat],
            "next_gamma": float(trained_theta[0]),
            "next_beta": float(trained_theta[1]),
            "plus_job_id": rows[0]["job_id"],
            "minus_job_id": rows[1]["job_id"],
        }
        spsa_log.append(step_row)
        print(json.dumps(step_row, indent=2))

    print(f"SPSA final theta: gamma={trained_theta[0]:.6f}, beta={trained_theta[1]:.6f}")

if hardware_training_evaluations:
    best_observed = min(hardware_training_evaluations, key=lambda row: row["mean_scaled_ising_energy"])
    best_observed_theta = np.array([best_observed["gamma"], best_observed["beta"]], dtype=float)
    print("Best observed training angle by measured scaled Ising energy:")
    print(json.dumps(best_observed, indent=2))
    display_metrics(hardware_training_evaluations)
else:
    best_observed = None
    best_observed_theta = wrap_angles([INITIAL_GAMMA, INITIAL_BETA])


## 7. Final High-Shot Evaluation

Set `RUN_FINAL_HARDWARE_EVAL = True` after training finishes. This submits one final IBM job at the best observed hardware-trained angle.


In [ ]:
final_hardware_evaluation = None

if not RUN_FINAL_HARDWARE_EVAL:
    print("RUN_FINAL_HARDWARE_EVAL is False. Set it to True after training to submit the final high-shot evaluation.")
else:
    if hardware_training_evaluations:
        final_theta = best_observed_theta
        final_source = "best_observed_training_theta"
    else:
        final_theta = wrap_angles([INITIAL_GAMMA, INITIAL_BETA])
        final_source = "initial_simulator_probe_theta"

    rows = run_hardware_evaluations(
        [final_theta],
        shots=FINAL_SHOTS,
        label="final_hardware_eval",
    )
    final_hardware_evaluation = rows[0]
    final_hardware_evaluation["theta_source"] = final_source

    random_projection_baseline = summarize_random_bitstring_projection_baseline(
        feasible_states,
        hardware_env,
        hardware_exact_rate,
        shots=FINAL_SHOTS,
        random_trials=256,
        seed=260630,
    )
    final_hardware_evaluation["random_bitstring_projection_baseline"] = random_projection_baseline
    final_hardware_evaluation["projected_optimum_count_ratio_vs_random_mean"] = (
        final_hardware_evaluation["projected_optimum_count"]
        / random_projection_baseline["projected_optimum_count_mean"]
        if random_projection_baseline["projected_optimum_count_mean"] > 0
        else None
    )
    print(json.dumps(final_hardware_evaluation, indent=2))


## 8. Save Hardware Training Evidence

This cell is safe. It writes whatever has been run so far to `qaoa_isac_hardware_training_results.json`.


In [ ]:
payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Real IBM hardware SPSA training for the 18-qubit full-binary QUBO QAOA bridge.",
    "hardware_training_claim": "Angles are optimized using measured IBM Sampler counts and scaled Ising energy. Projected AR is diagnostic only.",
    "scenario": {
        "U": HARDWARE_PARAMS.U,
        "G": HARDWARE_PARAMS.G,
        "S": HARDWARE_PARAMS.S,
        "Nt": HARDWARE_PARAMS.Nt,
        "Gamma_min": HARDWARE_PARAMS.Gamma_min,
        "seed": HARDWARE_SEED,
        "full_binary_qubits": hardware_env.n_qubits,
        "feasible_assignments": len(feasible_states),
        "exact_assignment": list(exact_state.assignment),
        "exact_sum_rate_mbps": hardware_exact_rate / 1e6,
    },
    "config": {
        "IBM_INSTANCE_set": IBM_INSTANCE is not None,
        "FIXED_BACKEND_NAME": FIXED_BACKEND_NAME,
        "TRAINING_SHOTS": TRAINING_SHOTS,
        "FINAL_SHOTS": FINAL_SHOTS,
        "SPSA_STEPS": SPSA_STEPS,
        "SPSA_SEED": SPSA_SEED,
        "SPSA_A0": SPSA_A0,
        "SPSA_C0": SPSA_C0,
        "SPSA_ALPHA": SPSA_ALPHA,
        "SPSA_GAMMA": SPSA_GAMMA,
        "initial_gamma": INITIAL_GAMMA,
        "initial_beta": INITIAL_BETA,
    },
    "transpilation_summary": transpilation_summary,
    "spsa_log": spsa_log,
    "hardware_training_evaluations": hardware_training_evaluations,
    "best_observed_training_evaluation": best_observed,
    "final_hardware_evaluation": final_hardware_evaluation,
}

OUTPUT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved {OUTPUT_JSON}")
print(json.dumps({
    "evaluations": len(hardware_training_evaluations),
    "has_final_evaluation": final_hardware_evaluation is not None,
    "output": str(OUTPUT_JSON),
}, indent=2))


## 9. How To Interpret Results

Good hardware-training evidence would look like this:

- final measured scaled Ising energy is lower than the initial-angle energy,
- projected optimum count improves versus the previous 18-qubit run or versus random projection,
- raw feasible count improves above `0/1024`, if possible,
- projected best AR remains `1.000`.

Still do not claim hardware quantum advantage unless the hardware-trained final evaluation beats the shot-matched random projection baseline by a meaningful margin and raw feasibility is not trivial.
